<a href="https://colab.research.google.com/github/amandaliram/pln_nlp_repository/blob/main/A09_RP06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Roteiro de Práticas 06:** Semântica e Vetores

Até este ponto, o pipeline de inteligência concentrou-se na "Faxina Digital" (Aulas 3 e 4) e na vetorização estatística (Aula 6), onde aprendemos a transformar texto em matrizes esparsas. No entanto, enquanto o TF-IDF é excelente para identificar a importância estatística de uma palavra, ele ainda falha em capturar o parentesco semântico entre "excelente" e "ótimo". Hoje, migramos para as representações densas para permitir que a máquina compreenda a profundidade do significado textual.
Ao final desta sessão, você deverá ser capaz de:


*   **Implementar** o treinamento de modelos Word2Vec via biblioteca gensim, distinguindo e escolhendo entre as arquiteturas CBOW e Skip-gram.
*   **Mensurar** a proximidade semântica entre termos críticos do domínio (ex: "preço", "valor", "custo") utilizando a matemática da Similaridade de Cosseno.
*   **Projetar** visualizações de alta dimensionalidade em planos 2D utilizando t-SNE, calibrando hiperparâmetros como perplexity para identificar clusters de opinião.
*   **Validar** a transição de representações esparsas e ortogonais para representações densas e latentes no pipeline de inteligência do e-commerce.

### **1. Introdução Teórica: O Mapa Geográfico das Palavras**
A migração da frequência bruta para o contexto semântico é uma mudança de paradigma estratégica na análise de sentimentos e extração de insights. No mercado atual, entender o "o quê" é dito é apenas o primeiro passo; o diferencial competitivo reside em entender o "como" os conceitos se interligam no imaginário do consumidor.

**1.1 As Limitações da Esparsidade**

Modelos como Bag-of-Words e TF-IDF (vistos na Aula 06) tratam palavras como símbolos isolados e ortogonais. No nano-corpus  que analisamos ("bateria excelente", "tela ruim"), o TF-IDF premiou o termo "excelente" com um peso alto (0.954) por ser localmente frequente e globalmente raro. Contudo, matematicamente, o vetor de "excelente" é tão distante de "bom" quanto de "geladeira". Essa falha em capturar a sinonímia ignora o contexto e gera matrizes gigantescas onde a relação entre os termos é invisível para o algoritmo.

**1.2 A Hipótese Distribucional**

O Word2Vec resolve esse impasse através da Hipótese Distribucional: "uma palavra é conhecida pela companhia que mantém". Em vez de olhar para a coleção global, treinamos redes neurais rasas para prever uma palavra a partir de sua vizinhança. Esse processo produz vetores densos, onde cada palavra é representada por um ponto em um espaço de múltiplas dimensões (embeddings). Aqui, a proximidade geométrica reflete a afinidade semântica; se "bateria" e "carregador" aparecem frequentemente no mesmo contexto de review, seus vetores serão ajustados para coexistirem na mesma região, mesmo que nunca apareçam na mesma sentença.


1.2.1 A Analogia do Mapa

Utilizamos a analogia do "Mapa Geográfico", onde palavras com funções similares ocupam o mesmo "bairro" vetorial. Palavras como "ruim", "péssimo" e "horrível" são coordenadas vizinhas em um cluster de insatisfação. Essa organização permite operações algébricas de significado, como a famosa equação Rei - Homem + Mulher = Rainha. No Social Listening, isso significa que podemos navegar pelo espaço latente para descobrir que, no vocabulário do seu cliente, o "bairro" da "entrega" está perigosamente próximo ao "bairro" da "frustração".

### **2. Exemplo 01**

In [1]:
%pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 31.8 MB/s eta 0:00:00


In [2]:
# Etapa1: Configuração e Treinamento

from gensim.models import Word2Vec
import pandas as pd

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("asadullahcreative/e-commerce-product-reviews")

print("Path to dataset files:", path)

100%|██████████| 274k/274k [00:00<00:00, 55.9MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/asadullahcreative/e-commerce-product-reviews/versions/1


In [4]:
# Preparação do corpus (Chain-of-Thought: carregar -> tokenizar -> janelar)
product_reviews = r"/root/.cache/kagglehub/datasets/asadullahcreative/e-commerce-product-reviews/versions/1/ecommerce-product-reviews.csv"

df = pd.read_csv(product_reviews)

sentences = [row.split() for row in df['review']]

In [5]:
# para conhecer as palavras que pertencem aos reviews
sentences

[['Law', 'simply', 'actually', 'race', 'it', 'run.'],
 ['Fish',
  'help',
  'admit',
  'group',
  'gun',
  'real.',
  'Trial',
  'road',
  'television',
  'analysis',
  'for',
  'produce.',
  'Factor',
  'where',
  'soon',
  'same',
  'child',
  'suggest.',
  'Eight',
  'reality',
  'however.',
  'Perform',
  'eye',
  'soldier',
  'accept',
  'owner',
  'nothing.'],
 ['Life',
  'standard',
  'score',
  'charge',
  'assume',
  'law.',
  'Station',
  'evidence',
  'control',
  'key',
  'minute',
  'once',
  'each.',
  'Or',
  'home',
  'worker',
  'purpose',
  'task',
  'trip',
  'air.'],
 ['And',
  'perform',
  'edge',
  'data',
  'sea',
  'first.',
  'Something',
  'reveal',
  'past',
  'find',
  'rest.',
  'Employee',
  'memory',
  'trouble',
  'enjoy',
  'southern',
  'practice.',
  'Statement',
  'interview',
  'dinner.'],
 ['Animal',
  'interview',
  'huge',
  'activity',
  'some',
  'health',
  'debate.',
  'Hold',
  'always',
  'western',
  'hair.',
  'Bed',
  'democratic',
  'af

In [7]:
# Treinamento do modelo Word2Vec
# (Certifique-se de que 'sentences' foi carregado corretamente na célula anterior)
model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=1, workers=4, sg=0)

# Proximidade entre conceitos: escolha duas palavras do modelo treinado:

#sim = model.wv.similarity('palavra01', 'palavra02')

sim = model.wv.similarity('later', 'century')

print(f"Similaridade Semântica: {sim:.4f}")

Similaridade Semântica: 0.9995


In [8]:
# Descoberta de vizinhos para Análise baseada em Aspecto
print("Aspectos relacionados a 'later':")
print(model.wv.most_similar('later', topn=10))

Aspectos relacionados a 'later':
[('officer', 0.9996572732925415), ('hit', 0.9996456503868103), ('wind', 0.9996399283409119), ('open', 0.9996352195739746), ('miss', 0.9996308088302612), ('activity', 0.9996260404586792), ('factor', 0.9996193051338196), ('drop', 0.9996184706687927), ('message', 0.9996168613433838), ('trade', 0.9996145963668823)]


### **3. Exemplo 2 - O Cenário: Palavras como Coordenadas**
Imagine que cada palavra do nosso vocabulário é um ponto em um espaço vetorial. O truque dos embeddings é que palavras com significados semelhantes estarão próximas umas das outras nesse espaço. Vamos ver isso na prática. Para esta aula, usaremos a biblioteca Gensim e um modelo de GloVe pré-treinado.

**3.1 Instale e configure o ambiente.**  

Primeiro, precisamos instalar a biblioteca Gensim e carregar um modelo de embedding pré-treinado. Para esta atividade, usaremos um modelo que foi treinado com 6 bilhões de tokens da Wikipedia e do Gigaword, o que garante um bom conhecimento de vocabulário geral.

In [9]:
%pip install gensim

In [10]:
# Vamos baixar um modelo de word embeddings pré-treinado (pode demorar um pouco)
import gensim.downloader as api
print("Baixando o modelo GloVe...")
# Você pode escolher outros modelos no site oficial, mas o GloVe é excelente para demonstração
glove_model_path = api.load("glove-wiki-gigaword-100", return_path=True)
print("Modelo GloVe baixado com sucesso!")

# Carrega o modelo na memória
from gensim.models import KeyedVectors
glove_model = KeyedVectors.load_word2vec_format(glove_model_path, binary=False)

print("\nModelo GloVe carregado e pronto para uso!")

Baixando o modelo GloVe...
[==================================================] 100.0% 128.1/128.1MB downloaded
Modelo GloVe baixado com sucesso!

Modelo GloVe carregado e pronto para uso!


**Calcule a similaridade entre palavras.**  

Agora que o modelo está carregado, podemos pedir a ele para encontrar a similaridade entre as palavras. A semelhança que ele encontra é baseada nos contextos em que as palavras aparecem, refletindo o seu significado.

In [11]:
# Calcule a similaridade cosseno entre duas palavras. O valor é de 0 a 1.
similaridade_king_queen = glove_model.similarity('king', 'queen')
print(f"Similaridade entre 'king' e 'queen': {similaridade_king_queen:.4f}")

similaridade_king_dog = glove_model.similarity('king', 'dog')
print(f"Similaridade entre 'king' e 'dog': {similaridade_king_dog:.4f}")

similaridade_man_woman = glove_model.similarity('man', 'woman')
print(f"Similaridade entre 'man' e 'woman': {similaridade_man_woman:.4f}")

Similaridade entre 'king' e 'queen': 0.7508
Similaridade entre 'king' e 'dog': 0.2951
Similaridade entre 'man' e 'woman': 0.8323


**Reflexão:** Observe que a similaridade entre "king" e "queen" é muito maior do que entre "king" e "dog". Isso demonstra que o modelo capturou a relação de realeza e gênero.

**3.2 Resolva analogias matemáticas.**
Esta é a parte mais fascinante. Com os vetores, podemos fazer operações matemáticas para desvendar as relações entre as palavras. Execute o código a seguir. A operação rei - homem + mulher irá nos dar um vetor que, no espaço semântico, está mais próximo da palavra "rainha".

In [12]:
# Encontre a palavra mais próxima da analogia: "rei - homem + mulher"
# A operação subtrai o vetor de 'homem' e adiciona o vetor de 'mulher'
resultado_analogia = glove_model.most_similar(positive=['queen', 'king'], negative=['man'])
print("Resultado da analogia: 'man' para 'woman' é como 'king' para...")
print(f"Resposta do modelo: {resultado_analogia[0]}")

Resultado da analogia: 'man' para 'woman' é como 'king' para...
Resposta do modelo: ('monarch', 0.635038435459137)


**Reflexão:** O que você nota no resultado? A resposta do modelo é "queen", mostrando que a relação de gênero (homem -> mulher) foi capturada e aplicada à relação de realeza (king -> queen).


### **4. Exemplo 3: A Importância do Contexto**
Neste código, estamos criando um modelo Word2Vec com um corpus extremamente pequeno, contendo apenas duas frases: "o cachorro está dormindo" e "o gato está dormindo". O modelo é instruído a aprender a representação vetorial de cada palavra, mas o único contexto em que ele vê as palavras "cachorro" e "gato" é o mesmo: ambas são precedidas por "o" e seguidas por "está dormindo".

In [13]:
# Importação a Biblioteca
from gensim.models import Word2Vec
# Word2Vec >>> criar os modelos de vetorização

# criação do Corpus
corpus = [
    ["o","cachorro","está","dormindo"],
    ["o","gato","está","dormindo"]
]

# Criando o modelo de vetor
model = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=1,sg=1)

# obtem o vetor da palavra
vector = model.wv['cachorro']

# calcula a similaridade de duas palavras
similarity = model.wv.similarity('cachorro','gato')

print("Similaridade entre 'cachorro' e 'gato': ",similarity)

Similaridade entre 'cachorro' e 'gato':  -0.0446171


**Reflexão:** Ao executar este código, a similaridade entre as palavras "cachorro" e "gato" será extremamente alta? Essa similaridade significa que o modelo "entendeu" que ambos são animais?

### **5. Exemplo 4: Impacto da Diversidade do Corpus**
Neste exemplo, o mesmo modelo Word2Vec é criado, mas agora com um corpus muito mais rico e diversificado, contendo dez frases. As palavras "cachorro" e "gato" aparecem em contextos diferentes, junto com outras palavras como "latindo", "brincando" e "inimigos". O corpus também introduz outras categorias, como objetos ("bola", "caneca") e conceitos abstratos ("céu", "lua").

In [14]:
from gensim.models import Word2Vec

corpus = [
    ["o", "cachorro", "está", "latindo", "no", "quintal"],
    ["o", "gato", "está", "miando", "no", "telhado"],
    ["o", "pássaro", "está", "voando", "no", "céu"],
    ["a", "bola", "está", "rolando", "no", "chão"],
    ["a", "criança", "está", "brincando", "com", "o", "cachorro"],
    ["o", "gato", "e", "o", "rato", "são", "inimigos"],
    ["a", "água", "está", "quente", "na", "caneca"],
    ["o", "sol", "está", "brilhando", "no", "céu"],
    ["a", "lua", "está", "cheia", "hoje"],
    ["a", "computador", "está", "ligado", "na", "mesa"]
]

model = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=1, sg=1)

# Calculando a similaridade entre palavras
print(f"Similaridade entre cachorro e gato       : {model.wv.similarity('cachorro', 'gato')}")
print(f"Similaridade entre cachorro e bola       : {model.wv.similarity('cachorro', 'bola')}")
print(f"Similaridade entre céu e lua             : {model.wv.similarity('céu', 'lua')}")
print(f"Similaridade entre computador e mesa     : {model.wv.similarity('computador', 'mesa')}")

Similaridade entre cachorro e gato       : 0.005066993180662394
Similaridade entre cachorro e bola       : 0.04625355452299118
Similaridade entre céu e lua             : 0.044282179325819016
Similaridade entre computador e mesa     : 0.026993313804268837


### **6. Usando outro modelo de pre treinado: GloVe no Colab**


**6.1 Configuração do Ambiente e Download do Modelo GloVe**  

Para esta atividade, você precisará baixar um modelo GloVe do repositório da Stanford NLP.
1. **Acesse o Repositório:** Visite o repositório da Stanford NLP.
2. **Baixe o Arquivo:** Encontre a seção "Pre-trained word vectors" e baixe o arquivo glove.2024.wikigiga.50d.zip. Este modelo é treinado em 11.9 bilhões de tokens da Wikipedia e do Gigaword, e representa cada palavra com um vetor de 50 dimensões.
3. **Extraia o Conteúdo:** Descompacte o arquivo glove.2024.wikigiga.50d.zip. Você encontrará um arquivo de texto chamado glove.2024.wikigiga.50d.txt.
4. **Salve no Google Drive:** Faça o upload do arquivo .txt para uma pasta de sua escolha no seu Google Drive.

**6.2 Carregando o Modelo no Google Colab**
Agora, vamos montar o seu Google Drive no Colab e carregar o modelo GloVe que você salvou.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Importe a biblioteca Gensim e defina o caminho do arquivo
from gensim.models import KeyedVectors

# Altere este caminho para o local onde você salvou o arquivo no seu Drive
glove_path = '/content/drive/MyDrive/FATEC/[2025.2][6] PLN/2 Atividades/glove.2024.wikigiga.50d.txt'

# Carrega o modelo de vetorização no arquivo especificado
# O parâmetro 'binary=False' indica que o arquivo está em formato de texto.
glove_model = KeyedVectors.load_word2vec_format(glove_path, binary=False, no_header=True)

**6.3 Explorando a Semântica das Palavras**  

Com o modelo carregado, você pode fazer operações de similaridade e analogia.

In [20]:
# Calcule a similaridade entre 'king' e 'queen'
similaridade = glove_model.similarity('king','queen')
print("Similaridade entre 'king' e 'queen': ", similaridade)

Similaridade entre 'king' e 'queen':  0.7507691


In [21]:
# Encontre as palavras mais próximas semanticamente de 'king'
palavras_proximas = glove_model.most_similar('king')
print("Palavras próximas de 'king': ", palavras_proximas)

Palavras próximas de 'king':  [('prince', 0.7682328820228577), ('queen', 0.7507690787315369), ('son', 0.7020888328552246), ('brother', 0.6985775232315063), ('monarch', 0.6977890729904175), ('throne', 0.6919989585876465), ('kingdom', 0.6811409592628479), ('father', 0.6802029013633728), ('emperor', 0.6712858080863953), ('ii', 0.6676074266433716)]


**Reflexão:** Observe que a similaridade entre 'king' e 'queen' é alta, pois o modelo aprendeu a relação de gênero e realeza. As palavras próximas a 'king' também se relacionam a este universo semântico, como 'prince', 'ruler' e 'queen', demonstrando a capacidade do modelo de capturar essas relações.

## **7.  Questionário de Síntese**

A reflexão técnica consolida a transição entre o código e a arquitetura de sistemas de linguagem.


1.   **Diferenciação de Pesos:** No TF-IDF (Aula 06), o peso de um termo é punido por sua onipresença global na coleção. No Word2Vec, como a frequência global de uma palavra impacta (ou não) sua posição no espaço vetorial em comparação com sua vizinhança imediata?
2.   **Análise de Similaridade e Ironia:** Se a similaridade de cosseno entre "entrega" e "atraso" for 0.98, mas o seu classificador de sentimentos indicar polaridade positiva em 40% desses casos, como o Word2Vec ajuda a identificar a ocorrência de ironia ou sarcasmo (tema da Aula 14)?
3.   **Compensação Dimensional:** Explique a relação entre o vector_size definido no treinamento e a eficácia do parâmetro perplexity no t-SNE. O que ocorre com a "nitidez" dos clusters se o tamanho do vetor for subdimensionado para um corpus muito vasto?

# 🧠 Respostas:

**1. Diferenciação de Pesos (TF-IDF vs Word2Vec)**
No TF-IDF, a frequência global age como um "punidor": palavras muito frequentes em todos os documentos perdem relevância estatística[cite: 3, 4]. No **Word2Vec**, a lógica muda completamente. A posição de uma palavra no espaço vetorial não é definida por quantas vezes ela aparece no total, mas sim pela sua **Hipótese Distribucional** — ou seja, por quem são seus "vizinhos imediatos"[cite: 3, 4].

Embora palavras muito frequentes (como "o", "que") apareçam em muitos contextos, arquiteturas robustas de Word2Vec aplicam uma técnica chamada *subsampling* (subamostragem) para ignorar termos excessivamente globais. Assim, a posição vetorial de uma palavra é moldada estritamente pela qualidade e especificidade do seu contexto local, e não pela sua contagem absoluta.

---

**2. Análise de Similaridade e Ironia**
Se "entrega" e "atraso" possuem similaridade 0.98, isso indica que no seu corpus elas são usadas quase nas mesmas frases (ex: "a *entrega* foi..." / "o *atraso* foi..."). Se o classificador de sentimentos aponta polaridade positiva (40%), temos um ruído semântico claro: pessoas estão elogiando um problema.

O **Word2Vec** ajuda a resolver a ironia porque ele constrói um "mapa geográfico" das palavras[cite: 3, 4]. Se clientes escrevem *"Parabéns pelo atraso, nota zero!"*, o Word2Vec vai aproximar o vetor de "atraso" de palavras como "parabéns" e "ótimo". Ao navegar por esse espaço latente, o algoritmo percebe que o "bairro" da frustração se fundiu com o "bairro" dos elogios[cite: 3, 4]. Quando modelos de Machine Learning (como redes neurais ou classificadores baseados nesses embeddings) leem esses vetores densos, eles aprendem a reconhecer esse "curto-circuito" estrutural como um padrão clássico de sarcasmo.

---

**3. Compensação Dimensional (vector_size vs t-SNE)**
O parâmetro `vector_size` define a "resolução" ou capacidade de memória do seu espaço semântico (geralmente usamos de 50 a 300 dimensões para capturar todas as nuances)[cite: 4].

Se você **subdimensionar** o vetor (ex: `vector_size=5` para um corpus de 10 milhões de palavras), você causará um "engarrafamento semântico". Não haverá eixos matemáticos suficientes para separar palavras levemente diferentes, forçando termos que não têm relação a ocuparem as mesmas coordenadas geométricas.

Quando você tentar visualizar isso no t-SNE (que converte muitas dimensões para 2D[cite: 3, 4]), o parâmetro de calibração `perplexity` será inútil. A "nitidez" dos clusters será completamente destruída, resultando em uma nuvem de pontos embaralhada e sem fronteiras claras, simplesmente porque a sobreposição semântica já aconteceu na raiz (no treinamento do Word2Vec).